### 3.5.11. Sequential Quadratic Programming (SQP)

$$
\min_{\mathbf{p}}\; \tfrac12 \mathbf{p}^\top \nabla^2_{xx} L(\mathbf{x}_k,\boldsymbol\lambda_k)\, \mathbf{p} + \nabla f(\mathbf{x}_k)^\top \mathbf{p}
\quad\text{s.t.}\quad
\nabla h(\mathbf{x}_k)^\top \mathbf{p} + h(\mathbf{x}_k) = \mathbf 0,
$$
with update $\mathbf{x}_{k+1} = \mathbf{x}_k + \mathbf{p}$.

**Explanation:**

Sequential quadratic programming (SQP) solves a nonlinear program by generating a sequence of quadratic-program subproblems: at each iterate it minimizes a quadratic model of the Lagrangian subject to linearized constraints. For an equality-constrained problem this is exactly Newton's method applied to the KKT system, so the step $(\mathbf{p},\boldsymbol\lambda_{k+1})$ is obtained by solving the linear KKT block
$$
\begin{bmatrix} \nabla^2_{xx} L & \nabla h \\ \nabla h^\top & 0 \end{bmatrix}
\begin{bmatrix} \mathbf{p} \\ \boldsymbol\lambda_{k+1} \end{bmatrix}
= -\begin{bmatrix} \nabla f \\ h \end{bmatrix}.
$$
Near a regular solution the iterates converge quadratically, mirroring the [equality-constrained Newton step](../09_Algorithms/35_equality_constrained_newton_step.ipynb).

**Numerical Example:**

Solve
$$
\min_{x_1,x_2}\; (x_1 - 2)^2 + (x_2 - 1)^2
\quad\text{subject to}\quad h(\mathbf{x}) = x_1^2 - x_2 = 0.
$$

Starting from $\mathbf{x}_0 = (1, 1)$ with $\lambda_0 = 0$, the Hessian of the Lagrangian is
$$
\nabla^2_{xx}L = \begin{bmatrix} 2 + 2\lambda & 0 \\ 0 & 2\end{bmatrix},
\qquad \nabla h = \begin{bmatrix} 2x_1 \\ -1\end{bmatrix}.
$$
Solving the KKT system for the step and updating drives the iterate toward the solution $\mathbf{x}^\star \approx (1.165, 1.358)$ with optimal value $\approx 0.825$ in a handful of iterations, while the constraint residual $x_1^2 - x_2$ shrinks quadratically toward zero.

In [1]:
import sympy as sp

x_1, x_2, lam = sp.symbols("x_1 x_2 lambda", real=True)
decision_variable = sp.Matrix([x_1, x_2])
objective = (x_1 - 2)**2 + (x_2 - 1)**2
constraint = sp.Matrix([x_1**2 - x_2])
lagrangian = objective - (sp.Matrix([lam]).T * constraint)[0]

gradient_f = sp.Matrix([sp.diff(objective, v) for v in decision_variable])
jacobian_h = constraint.jacobian(decision_variable)
hessian_L = sp.hessian(lagrangian, decision_variable)

iterate = {x_1: 1.0, x_2: 1.0, lam: 0.0}
for step in range(5):
    H = hessian_L.subs(iterate)
    A = jacobian_h.subs(iterate)
    rhs = -sp.Matrix.vstack(gradient_f.subs(iterate), constraint.subs(iterate))
    kkt = sp.Matrix.vstack(sp.Matrix.hstack(H, A.T), sp.Matrix.hstack(A, sp.zeros(1, 1)))
    delta = kkt.solve(rhs)
    iterate[x_1] += float(delta[0]); iterate[x_2] += float(delta[1]); iterate[lam] = float(delta[2])
    residual = float(constraint.subs(iterate)[0])
    print(f"iter {step+1}: x = ({iterate[x_1]:.6f}, {iterate[x_2]:.6f}), constraint residual = {residual:.2e}")

iter 1: x = (1.200000, 1.400000), constraint residual = 4.00e-02
iter 2: x = (1.157047, 1.336913), constraint residual = 1.84e-03
iter 3: x = (1.167438, 1.362804), constraint residual = 1.08e-04
iter 4: x = (1.164855, 1.356880), constraint residual = 6.68e-06
iter 5: x = (1.165503, 1.358397), constraint residual = 4.20e-07


**Equivalent casadi implementation:**

In [2]:
import casadi as ca

decision_variable = ca.SX.sym("x", 2)
objective = (decision_variable[0] - 2)**2 + (decision_variable[1] - 1)**2
constraint = decision_variable[0]**2 - decision_variable[1]

solver = ca.nlpsol("solver", "ipopt", {"x": decision_variable, "f": objective, "g": constraint},
                   {"ipopt.print_level": 0, "print_time": 0, "ipopt.sb": "yes"})
solution = solver(x0=[1, 1], lbg=0, ubg=0)

print("minimizer =", solution["x"])
print("minimum value =", float(solution["f"]))

minimizer = [1.16537, 1.35809]
minimum value = 0.824833705995427


**References:**

[📘 Nocedal, J., & Wright, S. J. (2006). *Numerical Optimization* (2nd ed.). Springer.](https://doi.org/10.1007/978-0-387-40065-5)  
[📘 Bertsekas, D. P. (1999). *Nonlinear Programming* (2nd ed.). Athena Scientific.](https://www.athenasc.com/nonlinbook.html)

---

[⬅️ Previous: Nonconvex Nonlinear Programs](./10_nonconvex_nonlinear_programs.ipynb) | [Next: Augmented Lagrangian Methods ➡️](./12_augmented_lagrangian_methods.ipynb)